<a href="https://colab.research.google.com/github/pollyaana/-Desktop-App/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22gensim_model_ipynb%22%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyldavis
import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim_models
import gzip
import requests
from io import BytesIO
from gensim.models import LdaModel
from gensim.models import CoherenceModel
import matplotlib.pyplot as plt
from gensim.corpora import Dictionary

# Получение токенизированных постов

In [ ]:
filename = 'state_tokens.csv.gz'
response = requests.get(f'https://raw.githubusercontent.com/componavt/sns4human/refs/heads/main/data/vk/posts/tokens/{filename}')
with gzip.GzipFile(fileobj=BytesIO(response.content), mode='rb') as f:
    tokens = pd.read_csv(f, delimiter=',', encoding='utf-8')['tokens']

data = tokens.apply(lambda x: x.split(' ')).tolist()


# Создание мешка слов
Преобразуем документы в векторизованную форму.
Мешок слов (BoW) — это  модель представления текста в виде числового вектора, где:

*    Учитывается наличие слов, но игнорируется их порядок (отсюда "мешок" — слова "перемешаны").

*   Каждому слову ставится в соответствие число (частота или бинарный флаг).




In [ ]:
dictionary = corpora.Dictionary(data)
corpus = [dictionary.doc2bow(text) for text in data]

num_topics = 8

# Тематическое моделирование

In [ ]:
lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15)

topics = [[(term, round(wt, 3))
               for term, wt in lda_model.show_topic(n)]
                   for n in range(0, lda_model.num_topics)]
topics_df = pd.DataFrame([[term for term, wt in topic]
                              for topic in topics],
                         index=['Тема '+str(t) for t in range(1, lda_model.num_topics+1)]).T
topics_df

,Тема 1,Тема 2,Тема 3,Тема 4,Тема 5,Тема 6,Тема 7,Тема 8
0,карелия,карелия,район,коренной,карелия,больница,школа,район
1,республика,карельский,карелия,предприятие,семья,медицинский,детский,сутки
2,национальный,республика,республика,карелия,память,центр,здание,посёлок
3,подробный,культура,рубль,производство,победа,строительство,проект,коронавирус
4,организация,желать,житель,развитие,великий,ремонт,петрозаводск,ситуация
5,проект,язык,бюджет,завод,ребёнок,помощь,спортивный,число
6,карельский,праздник,млн,компания,житель,республиканский,сад,находиться
7,язык,поздравлять,развитие,продукция,акция,оборудование,программа,карелия
8,конкурс,народный,программа,карельский,петрозаводск,карелия,посёлок,республика
9,общественный,коллектив,проект,хозяйство,отечественный,врач,ремонт,петрозаводск


# Оценка

Измерение тематических моделей с помощью связности.
Если тема представляет собой смесь определенных слов, то один из способов измерения семантической связности темы — вычислить совместную встречаемость между словами. То есть, как часто верхние слова в теме встречаются вместе в документах по сравнению с тем, как часто они встречаются независимо.

Оценка тематической модели — сложная тема без четкого количественного подхода, и она до сих пор обсуждается. Более высокая (или более низкая оценка в зависимости от меры) не обязательно означает более высокую качественную модель. То есть оценку, которую человек дал бы, глядя на тематические слова и насколько они интерпретируемы.

# Анализ текущих параметров модели

**Perplexity** — это часто используемая метрика для оценки моделей LDA. Она измеряет, насколько хорошо модель предсказывает новые данные. Более низкий показатель perplexity указывает на лучшую производительность. Вы можете рассчитать perplexity с помощью метода log_perplexity() модели LDA в gensim.

**Интерпретация:**

Низкая перплексия → модель хорошо предсказывает данные.

Высокая перплексия → модель плохо обобщает.

In [ ]:
lda_model.log_perplexity(corpus)    # Чем ближе к 0, тем лучше

-7.9755666732657104

In [ ]:
# Вычислить оценку согласованности
coherence_model_lda = CoherenceModel(model=lda_model, texts=data,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('Coherence Score:', coherence_lda)

Coherence Score: 0.580558482119647


# Попробуем повысить оценку

Мы удаляем редкие слова и общие слова на основе их частоты в документах . Ниже мы удаляем слова, которые встречаются менее чем в 20 документах или более чем в 50% документов. Попробуйте удалить слова только на основе их частоты или, может быть, объединить это с этим подходом.

Обрезка низко- и высокочастотных слов.

Одна из вещей, которую мы можем попробовать, — это отфильтровать редкие и распространенные токены.

In [ ]:
# Remove rare and common tokens.
from gensim.corpora import Dictionary

# Create a dictionary representation of the documents.
dictionary = Dictionary(data)

# Отфильтровать слова, которые встречаются менее чем в 20 документах или более чем в 40% документов.
dictionary.filter_extremes(no_below=20, no_above=0.4)
corpus = [dictionary.doc2bow(doc) for doc in data]

lda_model = LdaModel(corpus, num_topics=num_topics, id2word=dictionary, passes=15)

topics = [[(term, round(wt, 3))
               for term, wt in lda_model.show_topic(n)]
                   for n in range(0, lda_model.num_topics)]
topics_df = pd.DataFrame([[term for term, wt in topic]
                              for topic in topics],
                         index=['Тема '+str(t) for t in range(1, lda_model.num_topics+1)]).T
topics_df

,Тема 1,Тема 2,Тема 3,Тема 4,Тема 5,Тема 6,Тема 7,Тема 8
0,район,желать,праздник,национальный,предприятие,школа,сутки,проект
1,посёлок,поздравлять,акция,карельский,поддержка,коренной,ситуация,рубль
2,олонецкий,развитие,петрозаводск,язык,производство,ребёнок,число,строительство
3,поселение,здоровье,память,проект,коронавирус,детский,пациент,программа
4,село,рождение,житель,организация,завод,музей,находиться,бюджет
5,деревня,председатель,семья,подробный,мера,петрозаводский,проводиться,развитие
6,пряжинский,заместитель,великий,конкурс,бизнес,ребята,режим,район
7,житель,отмечать,казачий,культура,помощь,церковь,пневмония,средство
8,сельский,успех,карельский,общественный,предприниматель,центр,медицинский,млн
9,кондопожский,министр,победа,политика,компания,спортивный,инфекция,объект


In [ ]:
lda_model.log_perplexity(corpus)    # Чем ближе к 0, тем лучше

-7.319490678461557

In [ ]:
# Вычислить оценку согласованности
coherence_model_lda = CoherenceModel(model=lda_model, texts=data,
                                     dictionary=dictionary, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('Coherence Score:', coherence_lda)

Coherence Score: 0.6156025572378537


Оценка согласованности повысилась: 0.580558482119647 -> 0.6156025572378537

Фильтрование редких и частотных слов оказалось успешным

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
import sys
from gensim.utils import ClippedCorpus

def compute_coherence_values(corpus, dictionary, k, a, b):
    lda_model = LdaModel(corpus=corpus, num_topics=k,
                                id2word=dictionary, passes=10,
                                alpha=a, eta=b,
                                per_word_topics=True, random_state=42)
    coherence_model_lda = CoherenceModel(model=lda_model,
                                         texts=data,
                                         dictionary=dictionary, coherence='c_v')
    return coherence_model_lda.get_coherence()

grid = {}
grid['Validation_Set'] = {}

# Topics range
min_topics = 8
max_topics = 9
step_size = 1
topics_range = range(min_topics, max_topics+1, step_size)

# Alpha parameter
alpha = list(np.arange(0.01, 1, 0.3))
alpha.append('symmetric')
alpha.append('asymmetric')

# Beta parameter
beta = list(np.arange(0.01, 1, 0.3))
beta.append('symmetric')


# Validation sets
num_of_docs = len(corpus)
corpus_sets = [ClippedCorpus(corpus, int(num_of_docs*0.75)), corpus]
corpus_title = ['75% Corpus', '100% Corpus']

# results placeholder
model_results = {'Validation_Set': [],
                 'Topics': [],
                 'Alpha': [],
                 'Beta': [],
                 'Coherence': []
                }

# tqdm progress bar
pbar = tqdm(total=(len(beta)*len(alpha)*len(topics_range)*len(corpus_title)),
                 file=sys.stdout, colour='green')

### takes ~ 37 minutes
for i in range(len(corpus_sets)):
    for k in topics_range:    # iterate through validation corpuses
        for a in alpha:       # iterate through alpha values
            for b in beta:    # iterare through beta values
                # get the coherence score for the given parameters
                cv = compute_coherence_values(corpus=corpus_sets[i],
                                              dictionary=dictionary,
                                              k=k, a=a, b=b)
                # Save the model results
                model_results['Validation_Set'].append(corpus_title[i])
                model_results['Topics'].append(k)
                model_results['Alpha'].append(a)
                model_results['Beta'].append(b)
                model_results['Coherence'].append(cv)

                # update tqdm progress bar
                pbar.update(1)
                pbar.refresh()
pbar.close()

# save the results to a dataframe
df = pd.DataFrame(model_results)
# Находим максимальное значение в столбце 'Coherence'
max_coherence = df['Coherence'].max()

rows_with_max_coherence = df[df['Coherence'] == max_coherence]
rows_with_max_coherence

  0%|          | 0/120 [00:00<?, ?it/s]

,Validation_Set,Topics,Alpha,Beta,Coherence
102,100% Corpus,9,0.61,0.61,0.671176


In [ ]:
rows_with_max_coherence2 = df[df['Coherence'] > 0.62]
rows_with_max_coherence2

,Validation_Set,Topics,Alpha,Beta,Coherence
12,75% Corpus,8,0.61,0.61,0.636661
13,75% Corpus,8,0.61,0.91,0.641129
32,75% Corpus,9,0.01,0.61,0.637522
33,75% Corpus,9,0.01,0.91,0.637480
42,75% Corpus,9,0.61,0.61,0.643456
43,75% Corpus,9,0.61,0.91,0.637855
46,75% Corpus,9,0.91,0.31,0.638369
47,75% Corpus,9,0.91,0.61,0.634664
48,75% Corpus,9,0.91,0.91,0.634594
53,75% Corpus,9,symmetric,0.91,0.640062
